In [1]:
import json, pathlib, webbrowser
import numpy as np
import pandas as pd
import duckdb
import ipywidgets as w
from IPython.display import display, clear_output
import plotly.express as px
import plotly.graph_objects as go
from dotenv import load_dotenv
from irp.features.metrics import compute_metrics

_ROOT = pathlib.Path().resolve()
if _ROOT.name == "notebooks":
    _ROOT = _ROOT.parent

load_dotenv()
con = duckdb.connect(str(_ROOT / "data/irp.duckdb"), read_only=True)

fundamentals = {t: con.execute(f"SELECT * FROM {t}").df() for t in ("income", "balance", "cashflow")}
prices = con.execute("SELECT * FROM prices").df()

metrics_df = compute_metrics(fundamentals, prices, "annual", latest=True)

ticker_info = con.execute("""
    SELECT c.Ticker AS ticker, i.Sector AS sector, i.Industry AS industry
    FROM companies c
    LEFT JOIN industries i ON i.IndustryId = c.IndustryId
""").df()

metrics_df = metrics_df.merge(ticker_info, on="ticker", how="left")


def _theme():
    cfg = pathlib.Path("~/.config/Code/User/settings.json").expanduser()
    try:
        t = json.loads(cfg.read_text()).get("workbench.colorTheme", "")
        return "plotly_white" if "light" in t.lower() else "plotly_dark"
    except Exception:
        return "plotly_dark"


print(f"Loaded {len(metrics_df):,} tickers, {len(metrics_df.columns)} columns")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

/mnt/Dev/active_python_projects/investment_research_platform/.venv/lib/python3.13/site-packages/pandas/core/arraylike.py:402: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
/mnt/Dev/active_python_projects/investment_research_platform/.venv/lib/python3.13/site-packages/pandas/core/arraylike.py:402: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
/mnt/Dev/active_python_projects/investment_research_platform/.venv/lib/python3.13/site-packages/pandas/core/arraylike.py:402: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
/mnt/Dev/active_python_projects/investment_research_platform/.venv/lib/python3.13/site-packages/pandas/core/arraylike.py:402: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
/mnt/Dev/active_python_projects/investment_research_platform/.venv/lib/python3.13/site-p

Loaded 4,420 tickers, 50 columns


In [ ]:
import webbrowser

_META = {"ticker", "period", "sector", "industry"}
METRIC_COLS = sorted([c for c in metrics_df.columns if c not in _META and metrics_df[c].dtype.kind in "fc"])

# --- Sector / Industry ---
_all_sectors = ["(All)"] + sorted(metrics_df["sector"].dropna().unique().tolist())
sector_sel = w.SelectMultiple(
    options=_all_sectors, value=["(All)"], rows=6,
    description="Sector:", layout=w.Layout(width="220px"),
)
_all_industries = ["(All)"] + sorted(metrics_df["industry"].dropna().unique().tolist())
industry_sel = w.SelectMultiple(
    options=_all_industries, value=["(All)"], rows=6,
    description="Industry:", layout=w.Layout(width="220px"),
)

# --- Ticker ---
_excluded: set[str] = set()
ticker_filter = w.Text(placeholder="Filter tickers...", layout=w.Layout(width="180px"))
ticker_sel = w.SelectMultiple(options=[], value=[], rows=8, description="", layout=w.Layout(width="180px"))
select_all_btn = w.Button(description="Select All", layout=w.Layout(width="120px"))


def _refresh_tickers(*_):
    sel_s = [s for s in sector_sel.value if s != "(All)"]
    sel_i = [i for i in industry_sel.value if i != "(All)"]
    mask = pd.Series([True] * len(metrics_df), index=metrics_df.index)
    if sel_s:
        mask &= metrics_df["sector"].isin(sel_s)
    if sel_i:
        mask &= metrics_df["industry"].isin(sel_i)
    opts = sorted(metrics_df.loc[mask, "ticker"].unique().tolist())
    txt = ticker_filter.value.upper().strip()
    if txt:
        opts = [t for t in opts if txt in t]
    opts = [t for t in opts if t not in _excluded]
    ticker_sel.options = tuple(opts)


sector_sel.observe(_refresh_tickers, names="value")
industry_sel.observe(_refresh_tickers, names="value")
ticker_filter.observe(_refresh_tickers, names="value")
select_all_btn.on_click(lambda _: setattr(ticker_sel, "value", ticker_sel.options))
_refresh_tickers()

# --- Dimension toggle ---
dim_radio = w.RadioButtons(
    options=["2D", "3D"], value="2D",
    description="Mode:", layout=w.Layout(width="160px"),
)

# --- Axis dropdowns ---
DEFAULT_X = "P/E" if "P/E" in METRIC_COLS else METRIC_COLS[0]
DEFAULT_Y = "ROE" if "ROE" in METRIC_COLS else METRIC_COLS[1]
DEFAULT_Z = "Net Margin" if "Net Margin" in METRIC_COLS else METRIC_COLS[2]

x_dd = w.Dropdown(options=METRIC_COLS, value=DEFAULT_X, description="X:", layout=w.Layout(width="250px"))
y_dd = w.Dropdown(options=METRIC_COLS, value=DEFAULT_Y, description="Y:", layout=w.Layout(width="250px"))
z_dd = w.Dropdown(options=METRIC_COLS, value=DEFAULT_Z, description="Z:", layout=w.Layout(width="250px"))
z_row = w.HBox([z_dd])


def _toggle_z(*_):
    z_row.layout.display = "" if dim_radio.value == "3D" else "none"


dim_radio.observe(_toggle_z, names="value")
_toggle_z()

# --- Color / Size ---
color_dd = w.Dropdown(
    options=["sector", "industry"] + METRIC_COLS,
    value="sector",
    description="Color:",
    layout=w.Layout(width="250px"),
)
size_dd = w.Dropdown(
    options=["(none)"] + METRIC_COLS,
    value="Market Cap" if "Market Cap" in METRIC_COLS else "(none)",
    description="Size:",
    layout=w.Layout(width="250px"),
)

# --- Log scale (2D only) ---
log_x = w.Checkbox(value=False, description="Log X", indent=False, layout=w.Layout(width="100px"))
log_y = w.Checkbox(value=False, description="Log Y", indent=False, layout=w.Layout(width="100px"))
log_row = w.HBox([log_x, log_y])


def _toggle_log(*_):
    log_row.layout.display = "" if dim_radio.value == "2D" else "none"


dim_radio.observe(_toggle_log, names="value")

# --- Metric filters (AND-joined) ---
filter_rows: list[dict] = []
filters_box = w.VBox([])


def _add_filter(_=None, metric: str | None = None, vmin=None, vmax=None):
    metric_dd = w.Dropdown(
        options=METRIC_COLS, value=metric or METRIC_COLS[0],
        layout=w.Layout(width="230px"),
    )
    min_in = w.FloatText(value=vmin, description="min", layout=w.Layout(width="150px"),
                         style={"description_width": "initial"})
    max_in = w.FloatText(value=vmax, description="max", layout=w.Layout(width="150px"),
                         style={"description_width": "initial"})
    rm_btn = w.Button(description="✕", layout=w.Layout(width="34px"), button_style="warning")
    row_box = w.HBox([metric_dd, min_in, max_in, rm_btn])
    entry = {"box": row_box, "metric": metric_dd, "min": min_in, "max": max_in}
    filter_rows.append(entry)
    rm_btn.on_click(lambda _b: _remove_filter(entry))
    filters_box.children = tuple(e["box"] for e in filter_rows)


def _remove_filter(entry):
    filter_rows.remove(entry)
    filters_box.children = tuple(e["box"] for e in filter_rows)


add_filter_btn = w.Button(description="+ Metric filter", button_style="info",
                           layout=w.Layout(width="140px"))
add_filter_btn.on_click(_add_filter)

# --- Watchlist ---
_watchlist: list[str] = []
watchlist_sel = w.SelectMultiple(
    options=[], value=[], rows=10, description="",
    layout=w.Layout(width="160px"),
)
rm_wl_btn = w.Button(description="✕ Remove", button_style="warning", layout=w.Layout(width="100px"))
clear_wl_btn = w.Button(description="Clear all", button_style="danger", layout=w.Layout(width="90px"))
yf_btn = w.Button(description="Yahoo Finance", button_style="info", layout=w.Layout(width="120px"))


def _add_tickers(tickers):
    changed = False
    for t in tickers:
        if t not in _watchlist:
            _watchlist.append(t)
            changed = True
    if changed:
        watchlist_sel.options = tuple(sorted(_watchlist))


def _rm_from_watchlist(_=None):
    to_remove = list(watchlist_sel.value)
    for t in to_remove:
        if t in _watchlist:
            _watchlist.remove(t)
        _excluded.add(t)
    watchlist_sel.options = tuple(_watchlist)
    _refresh_tickers()


def _clear_watchlist(_=None):
    _watchlist.clear()
    watchlist_sel.options = ()


def _open_yahoo_finance(_=None):
    targets = list(watchlist_sel.value) or _watchlist
    for t in targets:
        webbrowser.open(f"https://finance.yahoo.com/quote/{t}/")


rm_wl_btn.on_click(_rm_from_watchlist)
clear_wl_btn.on_click(_clear_watchlist)
yf_btn.on_click(_open_yahoo_finance)

# --- Plot button + output ---
plot_btn = w.Button(description="Plot", button_style="primary", layout=w.Layout(width="100px"))
out = w.Output(layout=w.Layout(width="100%"))

# --- Layout ---
filter_col = w.VBox([w.HTML("<b>Sector</b>"), sector_sel, w.HTML("<b>Industry</b>"), industry_sel])
ticker_col = w.VBox([w.HTML("<b>Tickers</b>"), ticker_filter, ticker_sel, select_all_btn])
config_col = w.VBox([
    dim_radio,
    w.HTML("<b>Axes</b>"), x_dd, y_dd, z_row,
    w.HTML("<b>Style</b>"), color_dd, size_dd,
    log_row,
])
watchlist_col = w.VBox([
    w.HTML("<b>Selected</b>"),
    w.Label("Click / lasso to add"),
    watchlist_sel,
    rm_wl_btn,
    clear_wl_btn,
    yf_btn,
])

display(w.HBox([filter_col, ticker_col, config_col, watchlist_col]))
display(w.HTML("<b>Metric filters (AND-joined):</b>"))
display(filters_box)
display(w.HBox([add_filter_btn, plot_btn]))
display(out)

HTML(value='<b>Metric filters (AND-joined):</b>')

VBox()

Output(layout=Layout(width='100%'))

In [5]:
def _plot(_=None):
    tickers = list(ticker_sel.value) or list(ticker_sel.options)
    df = metrics_df[metrics_df["ticker"].isin(tickers)].copy().reset_index(drop=True)

    # Metric filters (AND-joined)
    for e in filter_rows:
        col = e["metric"].value
        if col not in df.columns:
            continue
        lo, hi = e["min"].value, e["max"].value
        if not pd.isna(lo):
            df = df[df[col] >= lo]
        if not pd.isna(hi):
            df = df[df[col] <= hi]

    x_col = x_dd.value
    y_col = y_dd.value
    z_col = z_dd.value
    color_col = color_dd.value
    size_col = size_dd.value if size_dd.value != "(none)" else None
    is_3d = dim_radio.value == "3D"

    required = [x_col, y_col] + ([z_col] if is_3d else [])
    df = df.dropna(subset=required).reset_index(drop=True)
    if size_col and size_col in df.columns:
        df = df[df[size_col].notna() & (df[size_col] > 0)].reset_index(drop=True)

    with out:
        clear_output(wait=True)
        if df.empty:
            print("No data to plot after filtering.")
            return

        show_text = len(df) <= 60

        hover_data = {"period": True, x_col: ":.3g", y_col: ":.3g"}
        if is_3d:
            hover_data[z_col] = ":.3g"
        if size_col and size_col in df.columns:
            hover_data[size_col] = ":.3g"

        kwargs = dict(
            data_frame=df,
            x=x_col,
            y=y_col,
            color=color_col if color_col in df.columns else None,
            custom_data=["ticker"],
            hover_name="ticker",
            hover_data=hover_data,
            template=_theme(),
        )
        if show_text:
            kwargs["text"] = "ticker"
        # Do NOT pass size= to px — we apply it manually below for correct scaling

        if is_3d:
            fig = px.scatter_3d(
                **kwargs,
                z=z_col,
                title=f"{y_col} vs {x_col} vs {z_col}  ({len(df)} tickers)",
            )
            fig.update_traces(marker_opacity=0.75)
            if show_text:
                fig.update_traces(textfont_size=8)
            fig.update_layout(height=900, autosize=True)
        else:
            fig = px.scatter(
                **kwargs,
                log_x=log_x.value,
                log_y=log_y.value,
                title=f"{y_col} vs {x_col}  ({len(df)} tickers)",
            )
            fig.update_traces(marker_opacity=0.7)
            if show_text:
                fig.update_traces(textposition="top center", textfont_size=8)
            fig.update_layout(height=900, autosize=True)

        fw = go.FigureWidget(fig)

        # Apply size manually: normalize to [6, 50] pixel diameters, consistent across all traces
        if size_col and size_col in df.columns:
            v = df[size_col].values.astype(float)
            p5, p95 = np.nanpercentile(v, 5), np.nanpercentile(v, 95)
            v_clipped = np.clip(v, p5, p95)
            v_min, v_max = v_clipped.min(), v_clipped.max()
            v_norm = (v_clipped - v_min) / (v_max - v_min + 1e-9)
            pixel_sizes = 6 + v_norm * 44  # diameter 6px (smallest) to 50px (largest)
            ticker_to_size = dict(zip(df["ticker"].values, pixel_sizes))
            for trace in fw.data:
                if trace.customdata is not None and len(trace.customdata):
                    sizes = [ticker_to_size.get(cd[0], 8) for cd in trace.customdata]
                    trace.update(marker=dict(size=sizes, sizemode="diameter", sizeref=1))
        else:
            for trace in fw.data:
                trace.update(marker_size=7)

        def _on_click(trace, points, selector):
            _add_tickers([trace.customdata[i][0] for i in points.point_inds])

        def _on_select(trace, points, selector):
            _add_tickers([trace.customdata[i][0] for i in points.point_inds])

        for trace in fw.data:
            trace.on_click(_on_click)
            trace.on_selection(_on_select)

        display(fw)


plot_btn.on_click(_plot)